# Lab 01: The Reflection Pattern for Evidence-Bounded Reasoning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab01_reflection_pattern/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

In this lab an AI model writes a first **digital forensics incident report** from a set of phone
**artifacts** (the files and records pulled from the phone), then a second pass **critiques** that
report and the model **revises** it. This
generate -> critique -> revise loop is the **Reflection Pattern**.

Your job is not to trust the model. It is to **inspect each stage**, trace every claim back to the
evidence, and decide whether the final report stays inside what the artifacts actually support. The
agent is a critique aid. The forensic judgment stays with you.

You will build the loop **by hand first** (Part A), then run the **same workflow through a reusable
`ReflectionAgent` class** (Part B).

> **Before this notebook:** if your instructor assigned the 35-minute group evidence activity,
> finish it first and keep your answers. You will compare them against the model's output.

## Part 0: One-Time Setup

Run the next three cells in order. They install one library, connect you to a **free hosted model**
(Groq), and write the synthetic case files next to the notebook. No local install is required.

### Step 0.1: Install the client library

**Purpose:** install the `openai` client, the one small library that lets the notebook talk to the
model (Groq uses the same interface).  
**What you'll see:** a short pip log. Warnings are normal.

In [ ]:
# Purpose: install the one library this notebook needs, inside the hosted session only (not on your PC).
%pip install -q openai

### Step 0.2: Connect to a free hosted model (Groq)

**Purpose:** get a free **API key** (a private password that lets the notebook use the model) and build a client. We use **Groq**, which runs open models and
needs no credit card.

1. Open **https://console.groq.com/keys** and sign in (Google or GitHub works).
2. Click **Create API Key**, copy it.
3. Run the cell below and paste it when asked. The key is hidden as you type and is **not saved**
   into the notebook.

**What you'll see:** `Model OK: ...` once the connection works.

In [ ]:
# Purpose: collect the Groq key privately and build an OpenAI-compatible client pointed at Groq.
import os        # os: lets the notebook read saved settings, such as where your key is kept
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: the client that carries your messages to the model (Groq speaks the same language)

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

BASE_URL = 'https://api.groq.com/openai/v1'
# Free and strong. If this name ever errors, run the model-list cell below and pick another.
MODEL = 'llama-3.3-70b-versatile'

client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'])

try:
    probe = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': 'Reply with one word: ready'}],
    )
    print('Model OK:', MODEL, '->', probe.choices[0].message.content.strip())
except Exception as err:
    print('Could not use', MODEL)
    print('Reason:', err)
    print('\nModels available on your key:')
    try:
        for m in client.models.list().data:
            print('  -', m.id)
    except Exception as err2:
        print('  (could not list models:', err2, ')')
    print('\nSet MODEL above to one of these (for example llama-3.1-8b-instant) and re-run.')

### Step 0.3: Write the synthetic case files

**Purpose:** create a local `data/` folder holding the six evidence artifacts for case
`DF-2026-017`. These are synthetic, built for teaching. The same files live in the repository.  
**What you'll see:** a list of the files written. Existing files are not overwritten.

In [ ]:
# Purpose: create the case files on disk so the notebook has everything it needs, with no download.
from pathlib import Path  # Path: a readable way to refer to files and folders by name

ARTIFACTS = {'artifact_manifest.json': '{\n  "case_id": "DF-2026-017",\n  "device": {\n    "platform": "Android",\n    "model": "Pixel 7",\n    "os_version": "Android 14"\n  },\n  "acquisition": {\n    "type": "full_file_system",\n    "tool": "Cellebrite UFED 7.64",\n    "acquired_at_utc": "2026-02-18T14:32:11Z",\n    "original_image_sha256": "a5f30f7f6a6c95c17117d4ea03f2a618f9380ca379f6f31df96ab53ac49f58a8"\n  },\n  "analysis_timezone": "America/New_York",\n  "incident_window_utc": {\n    "start": "2026-02-14T20:00:00Z",\n    "end": "2026-02-16T04:00:00Z"\n  },\n  "note": "Synthetic classroom dataset for reflection-pattern demo"\n}\n', 'file_events.csv': 'timestamp_utc,path,event_type,sha256\n2026-02-15T01:05:22Z,/storage/emulated/0/Download/customers_q1.csv,created,531ec9ebf8f4492754f16b1437f1824995c8bf4ee2f6be74d4f2f13fdf3f2ab5\n2026-02-15T01:06:12Z,/storage/emulated/0/Download/customers_q1.csv,accessed,531ec9ebf8f4492754f16b1437f1824995c8bf4ee2f6be74d4f2f13fdf3f2ab5\n2026-02-15T01:07:41Z,/storage/emulated/0/Android/data/com.telegram.messenger/files/Telegram/Documents/customers_q1.csv,copied,531ec9ebf8f4492754f16b1437f1824995c8bf4ee2f6be74d4f2f13fdf3f2ab5\n2026-02-15T01:08:59Z,/storage/emulated/0/DCIM/Camera/IMG_20260215_010859.jpg,created,2a427f455903f62185111b45e22f4f55e7e5a3f7ed4d90dd357b95a8619a185d\n2026-02-15T01:12:04Z,/storage/emulated/0/Download/customers_q1.csv,deleted,531ec9ebf8f4492754f16b1437f1824995c8bf4ee2f6be74d4f2f13fdf3f2ab5\n', 'app_db_messages.csv': 'timestamp_utc,app,contact,direction,text_excerpt\n2026-02-15T01:07:58Z,Telegram,@external_consult,OUT,"sending that sheet now"\n2026-02-15T01:08:15Z,Telegram,@external_consult,OUT,"delete after download"\n2026-02-15T01:09:44Z,Gmail,personal_mail,OUT,"see attached export"\n2026-02-15T01:11:51Z,Slack,#sales-ops,IN,"Need customer list for Monday forecast"\n', 'network_events.csv': 'timestamp_utc,domain,ip,bytes_out,process\n2026-02-15T01:07:54Z,api.telegram.org,149.154.167.220,285901,org.telegram.messenger\n2026-02-15T01:09:39Z,smtp.gmail.com,142.250.114.108,194221,com.google.android.gm\n2026-02-15T01:10:08Z,drive.google.com,142.250.191.142,16992,com.android.chrome\n2026-02-15T01:11:02Z,intranet.corp.local,10.40.22.18,8422,com.microsoft.teams\n', 'location_events.csv': 'timestamp_utc,lat,lon,source\n2026-02-15T00:58:00Z,40.71274,-74.00595,gps\n2026-02-15T01:06:00Z,40.71275,-74.00593,wifi\n2026-02-15T01:12:00Z,40.71273,-74.00590,cell\n', 'chain_of_custody.csv': 'timestamp_utc,handler,action,notes\n2026-02-18T14:32:11Z,Investigator_A,acquired_image,"UFED acquisition to encrypted evidence drive"\n2026-02-18T15:04:29Z,Investigator_A,computed_hash,"SHA-256 verified against acquisition log"\n2026-02-19T09:10:03Z,Analyst_B,created_working_copy,"Mounted read-only image; exported SQLite and CSV artifacts"\n'}

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
for name, text in ARTIFACTS.items():
    fp = data_dir / name
    if not fp.exists():
        fp.write_text(text)
    print(('wrote ' if fp.stat().st_size else 'exists ') + str(fp))
print('\nData folder ready:', data_dir.resolve())

## The Case in One Minute

**Case `DF-2026-017`** - a company Google Pixel 7 is reviewed after a possible customer-data leak
before an employee left. During the incident window the file `customers_q1.csv` appears in
Downloads, is opened, is copied into Telegram's storage, and is then deleted, with Telegram, Gmail,
and network activity around the same time.

| UTC time | What the artifacts record |
|---|---|
| 01:05:22 | `customers_q1.csv` created in Downloads |
| 01:06:12 | same file accessed |
| 01:07:41 | same file hash copied into Telegram's `Documents` folder |
| 01:07:54 | outbound traffic to `api.telegram.org` |
| 01:07:58 | Telegram message out: "sending that sheet now" |
| 01:08:15 | Telegram message out: "delete after download" |
| 01:09:39 | outbound SMTP traffic via Gmail |
| 01:09:44 | Gmail message out: "see attached export" |
| 01:12:04 | `customers_q1.csv` deleted from Downloads |

These events are **consistent with** an attempt to share the file. They do **not** by themselves
prove the file reached anyone. Keeping that line clear is the whole point of this lab.

## Part A: Build the Reflection Loop by Hand

We will do the three moves of the pattern one at a time: **generate** a draft, **reflect** on it,
then **revise** it.

### Step A.1: Load the artifacts into one text bundle

**Purpose:** read the evidence files and pack them into a single string we can hand to the model.  
**Input:** the files in `data/`.  
**Output:** `artifact_bundle`, the text the model will reason over.

In [ ]:
# Purpose: build the evidence bundle the model will analyze.
from pathlib import Path
from IPython.display import display_markdown  # render the model's report as formatted text, not raw text

data_dir = Path('data')

def excerpt(filename, n_lines=8):
    lines = (data_dir / filename).read_text().strip().splitlines()
    return '\n'.join(lines[:n_lines])

artifact_bundle = f'''
artifact_manifest.json
{(data_dir / 'artifact_manifest.json').read_text()}

file_events.csv
{excerpt('file_events.csv')}

app_db_messages.csv
{excerpt('app_db_messages.csv')}

network_events.csv
{excerpt('network_events.csv')}

location_events.csv
{excerpt('location_events.csv')}

chain_of_custody.csv
{excerpt('chain_of_custody.csv')}
'''
print(artifact_bundle[:600], '...')

### Step A.2: Set the generator's role and ask for a report

**Purpose:** tell the model it is a forensic analyst that must cite evidence, then give it the task.  
**Output:** a `generation_chat_history` list, the running memory of the analyst role.

In [ ]:
# Purpose: start the generator conversation with a system role and the analysis request.
generation_chat_history = [
    {
        'role': 'system',
        'content': (
            'You are a digital forensics analyst writing a defensible preliminary incident report. '
            'Use only the provided artifacts. Cite concrete evidence in each claim (timestamp plus '
            'artifact source). Separate observation from inference. If the user provides critique, '
            'respond with a revised version of your previous report.'
        ),
    },
    {
        'role': 'user',
        'content': (
            'Analyze this mobile forensic case and produce a preliminary incident report.\n'
            'Return: (1) a timeline, (2) a primary hypothesis, (3) two alternative hypotheses, '
            '(4) a confidence level (low / medium / high) per claim, (5) an explicit evidence map.\n\n'
            f'Artifacts:\n{artifact_bundle}'
        ),
    },
]
print('Generator primed with', len(generation_chat_history), 'messages.')

### Step A.3: Generate the first draft

**Purpose:** get the model's first report.  
**What you'll see:** a confident report. Watch for claims that go further than the evidence.

In [ ]:
# Purpose: produce draft v1 and keep it in the generator's memory so it can revise later.
incident_report_draft = client.chat.completions.create(
    model=MODEL,
    messages=generation_chat_history,
).choices[0].message.content

generation_chat_history.append({'role': 'assistant', 'content': incident_report_draft})
display_markdown(incident_report_draft, raw=True)

### Step A.4: Reflect - have the model critique the draft

**Purpose:** create a separate **reviewer** role whose only job is to catch overclaiming, claims with
no artifact to back them, and vague wording about how sure the report is.  
**Output:** a `critique`. If the reviewer finds nothing material it must output exactly `[[OK]]`.

In [ ]:
# Purpose: set up the reviewer role and feed it the draft to critique.
reflection_chat_history = [
    {
        'role': 'system',
        'content': (
            'You are a senior digital forensics reviewer. Critique the report for evidentiary rigor. '
            'For each issue, say why it is a risk and how to fix it. Focus on provenance gaps, '
            'timestamp inconsistencies, chain-of-custody concerns, and overclaiming (for example '
            'claiming delivery or receipt that the artifacts do not confirm). '
            'If there are no material issues, output exactly [[OK]].'
        ),
    },
    {'role': 'user', 'content': f'Please review this forensic report:\n\n{incident_report_draft}'},
]

critique = client.chat.completions.create(
    model=MODEL,
    messages=reflection_chat_history,
).choices[0].message.content
display_markdown(critique, raw=True)

### Step A.5: Revise - feed the critique back to the analyst

**Purpose:** hand the critique to the generator, which still remembers its draft and the evidence,
and ask for a revision.  
**What you'll see:** a more careful report. Compare its language to the first draft.

In [ ]:
# Purpose: add the critique to the generator's memory and produce the revised report.
generation_chat_history.append({
    'role': 'user',
    'content': f'Revise your previous forensic report using this critique.\n\nCritique:\n{critique}',
})

revised_incident_report = client.chat.completions.create(
    model=MODEL,
    messages=generation_chat_history,
).choices[0].message.content
display_markdown(revised_incident_report, raw=True)

### Step A.6: Iterate when needed

In a full loop the revised report goes back to the reviewer, and the cycle repeats until the
critique returns `[[OK]]` or a step limit is reached. Part B does that looping for you.

## Part B: Run the Same Workflow with a Reusable `ReflectionAgent`

Part A showed every move. Part B wraps the exact same loop into one class so you can run it with a
single call. The class below mirrors `src/agentic_patterns/reflection_pattern/` in the repository,
and is adapted from The Neural Maze's agentic-patterns-course (MIT License).

### Step B.1: Define the agent

**Purpose:** load the reusable reflection components. Read the `run()` method: it is Part A's
generate -> reflect -> revise loop, with a `[[OK]]` stop check and a bounded chat history.

In [ ]:
# Purpose: the packaged Reflection Pattern. Same logic as Part A, wrapped for reuse.
# Mirrors src/agentic_patterns/reflection_pattern/ in the repo.
# Adapted from The Neural Maze agentic-patterns-course (MIT License), (c) 2024 The Neural Maze.

# ----------------------------------------------------------------- completions utils
def completions_create(client, messages, model):
    response = client.chat.completions.create(messages=list(messages), model=model)
    return str(response.choices[0].message.content)


def build_prompt_structure(prompt, role, tag=""):
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


def update_chat_history(history, msg, role):
    history.append(build_prompt_structure(prompt=msg, role=role))


class ChatHistory(list):
    def __init__(self, messages=None, total_length=-1):
        if messages is None:
            messages = []
        super().__init__(messages)
        self.total_length = total_length

    def append(self, msg):
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)

    def show_messages(self):
        if not self:
            print("ChatHistory is empty.")
            return
        for idx, message in enumerate(self, start=1):
            if isinstance(message, dict):
                role = message.get("role", "unknown")
                content = message.get("content", "")
                preview = content if len(content) < 160 else content[:157] + "..."
                print(f"{idx}. [{role}] {preview}")
            else:
                print(f"{idx}. {message}")


class FixedPrefixChatHistory(ChatHistory):
    """Keeps an initial prefix pinned (system + original task) while newer messages rotate."""

    def __init__(self, messages=None, total_length=-1, pinned_prefix_len=1):
        super().__init__(messages, total_length)
        self.pinned_prefix_len = pinned_prefix_len

    def append(self, msg):
        if self.total_length != -1 and len(self) == self.total_length:
            self.pop(self.pinned_prefix_len)
        list.append(self, msg)


# ----------------------------------------------------------------- the agent
BASE_GENERATION_SYSTEM_PROMPT = """
Your task is to Generate the best content possible for the user's request.
If the user provides critique, respond with a revised version of your previous attempt.
You must always output the revised content.
"""

BASE_REFLECTION_SYSTEM_PROMPT = """
You are tasked with generating critique and recommendations to the user's generated content.
If the user content has something wrong or something to be improved, output a list of recommendations
and critiques. If the user content is correct and there's no recommendations and critiques, you should
output exactly the string: [[OK]], otherwise, you must output [[not OK]].
"""


class ReflectionAgent:
    """Generates a draft, critiques it, and revises it, looping until the critique says [[OK]]."""

    def __init__(self, model="llama-3.3-70b-versatile", client=None):
        if client is None:
            raise ValueError(
                "ReflectionAgent requires a `client` instance that supports "
                "`chat.completions.create(...)`."
            )
        self.client = client
        self.model = model

    def _request_completion(self, history, verbose=0, log_title="COMPLETION"):
        output = completions_create(self.client, history, self.model)
        if verbose > 0:
            print(f"\n\n{log_title}\n\n{output}")
        return output

    def generate(self, generation_history, verbose=0):
        return self._request_completion(generation_history, verbose, log_title="GENERATION")

    def reflect(self, reflection_history, verbose=0):
        return self._request_completion(reflection_history, verbose, log_title="REFLECTION")

    def run(self, user_msg, generation_system_prompt="", reflection_system_prompt="",
            n_steps=3, verbose=0):
        generation_system_prompt += BASE_GENERATION_SYSTEM_PROMPT
        reflection_system_prompt += BASE_REFLECTION_SYSTEM_PROMPT

        generation_history = FixedPrefixChatHistory(
            [
                build_prompt_structure(prompt=generation_system_prompt, role="system"),
                build_prompt_structure(prompt=user_msg, role="user"),
            ],
            total_length=4,       # system + task + (draft) + (critique)
            pinned_prefix_len=2,  # pin system + original task
        )
        reflection_history = FixedPrefixChatHistory(
            [build_prompt_structure(prompt=reflection_system_prompt, role="system")],
            total_length=4,
            pinned_prefix_len=1,
        )

        generation = ""
        for step in range(n_steps):
            if verbose > 0:
                print(f"\n{'=' * 50}\nSTEP {step + 1}/{n_steps}\n{'=' * 50}")

            generation = self.generate(generation_history, verbose=verbose)
            update_chat_history(generation_history, generation, "assistant")
            update_chat_history(reflection_history, generation, "user")

            critique = self.reflect(reflection_history, verbose=verbose)

            if "[[OK]]" in critique:
                update_chat_history(reflection_history, critique, "assistant")
                print("\nStop sequence found. Stopping the reflection loop.\n")
                break

            update_chat_history(reflection_history, critique, "assistant")
            update_chat_history(generation_history, critique, "user")

        return generation

### Step B.2: Create the agent and run it

**Purpose:** give the agent the same forensic task and let it loop on its own.  
**What you'll see:** each GENERATION and REFLECTION step, then the loop stops when the reviewer is
satisfied. `n_steps` caps the number of cycles so a free-tier key is never overused.

In [ ]:
# Purpose: run the packaged reflection loop on the same case.
agent = ReflectionAgent(model=MODEL, client=client)

generation_system_prompt = (
    'You are a digital forensics analyst writing a defensible incident report. '
    'Use only user-provided evidence, cite timestamps and artifact sources, and revise when '
    'critique is provided.'
)
reflection_system_prompt = (
    'You are a senior digital forensics reviewer. Flag overclaiming, provenance gaps, and weak '
    'uncertainty language. Output exactly [[OK]] when there is nothing material to improve.'
)

final_report = agent.run(
    user_msg=(
        'Analyze this mobile forensic case and produce a preliminary incident report.\n'
        'Return: (1) a timeline, (2) a primary hypothesis, (3) two alternative hypotheses, '
        '(4) a confidence level per claim, (5) an explicit evidence map.\n\n'
        f'Artifacts:\n{artifact_bundle}'
    ),
    generation_system_prompt=generation_system_prompt,
    reflection_system_prompt=reflection_system_prompt,
    n_steps=3,
    verbose=1,
)

### Step B.3: Inspect the final report
**Purpose:** read the report the loop settled on.

In [ ]:
# Purpose: show the final, reviewed report.
display_markdown(final_report, raw=True)

## Your Evidence Check

Answer in the cell below, in your own words. Base every answer on the artifacts and the outputs
above, not on general knowledge.

1. Find one claim in the **first draft** that was too strong. Which artifacts support a narrower
   claim instead?
2. Pick one claim in the **revised report**. Give the artifact source and UTC timestamp that back
   it, then state one limit on that claim.
3. Did the critique improve the report's **uncertainty language**? Quote one phrase that changed
   from draft to revision.
4. What forensic judgment must **stay with a human**, even after the loop ends?

In [ ]:
answers = {
    'q1_overclaim': '... your answer ...',
    'q2_supported_claim': '... artifact + timestamp + limit ...',
    'q3_uncertainty_change': '... phrase before -> after ...',
    'q4_human_judgment': '... your answer ...',
}
for k, v in answers.items():
    print(k, '->', v)

## Submit

Save the notebook with the draft, critique, revised report, Part B output, and your four answers
filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.
- Hand in the `.ipynb` as your instructor directs.

---

*Case data and the agentic-pattern course structure are adapted from*
*[frankwxu/agentic-AI4-forensics](https://github.com/frankwxu/agentic-AI4-forensics). Agent code is*
*adapted from [The Neural Maze's agentic-patterns-course](https://github.com/neural-maze/agentic-patterns-course) (MIT License).*